# 03 — When particles interact (Student workbook)

**Time:** 55 minutes
**Question:** can a poor fit be repaired by changing particle shape, or is
inter-particle physics missing?

Start with the simplest plausible particle model. Use the residual pattern as a
clue before adding complexity.

> **Lecture connection**
>
> - Slide 9, *Real and Q-space*: "the blob sets the envelope, the arrangement
>   sets the sharp structure". This is the $P(Q)\cdot S(Q)$ split.
> - Slide 24, *The structure factor, S(Q)*: hard-sphere repulsion suppresses
>   low-Q intensity and creates a correlation peak near $Q^* \approx 2\pi/\sigma$.
> - Slide 19, *Guinier analysis*: a low-Q **downturn** is the warning sign for
>   repulsive interactions. An upturn suggests aggregation.
> - Slide 23, *Name that shape!*, panel D: concentrated spheres. Dilute the
>   sample, and if the peak disappears it was $S(Q)$.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
filename = DATA / 'interacting_spheres.csv'
values = np.genfromtxt(filename, delimiter=',', names=True)
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(values['Q'], values['I'], yerr=values['dI'], fmt='o', ms=3)
ax.set(xscale='log', yscale='log', xlabel=r'$q$ / Å$^{-1}$', ylabel=r'$I(q)$')
ax.grid(alpha=0.25)
plt.show()


### Look before you fit

Compare this curve with the dilute sphere from notebook 01.

1. What happens at the lowest q: plateau, downturn or upturn? What does
   slide 19 say each one means?
2. Find the q position of the bump, $Q^*$. Convert it to a real-space
   distance, $2\pi/Q^*$. Is that distance closer to a particle *radius*, a
   *diameter*, or neither? (Slide 24: the first peak sits near
   $Q^* \approx 6/\sigma$, where $\sigma = 2R_{hs}$ is the contact distance.)
3. Is the bump a feature of one particle or of the arrangement of many?
   Name the experiment that would tell you (slide 9's discussion prompt).

## Exercise 1 — Deliberately incomplete model

From the previous notebooks, build a dilute-sphere fit without copying a
complete solution. Locate the structured residual region. Why can changing
radius or scale not independently repair it?

*Hint:* the lecture's "four knobs" (slide 18). Concentration, contrast and
scale only move the whole curve **up or down**. Size moves features **left or
right** and also changes $I(0) \propto V$. Neither move can lower the
intensity at low q while raising it at $Q^*$. Also predict what a Guinier
analysis of these data would give for the size (slide 24: "Guinier analysis on
a concentrated sample under-estimates the size").

In [ ]:
# Build the simplest plausible explanation: a dilute sphere.
# Keep both SLD values fixed and vary radius, scale, and background.

# YOUR CODE HERE
raise NotImplementedError('Build and fit the dilute-sphere model')


## Exercise 2 — Add excluded-volume interactions

`set_structure_factor('hardsphere')` combines the selected form factor with a
hard-sphere interaction model. Calling `get_params()` afterward reveals the
additional parameters:

- `volfraction`: fraction of the sample volume occupied by interacting spheres;
- `radius_effective`: interaction distance, which may include a solvation layer.

Fit both independently. The mode-control parameters should remain fixed.

**What is being multiplied in.** This is the full master equation of slide 18,
$I(Q) = \phi V \Delta\rho^2 P(Q)\,S(Q) + \mathrm{bkg}$, now with $S(Q) \neq 1$.
$S(Q)$ is the Fourier transform of the pair correlation function $g(r)$:
"the form factor of the neighbours" (slide 24). The `hardsphere` model uses
the Percus–Yevick closure for particles that cannot overlap
([SasView `hardsphere`](https://www.sasview.org/docs/user/models/hardsphere.html)). Two limits from the lecture to check against your fit:

- $S(0) = (1-\eta)^4/(1+2\eta)^2$. It falls from 0.92 at $\eta = 0.01$ to
  0.09 at $\eta = 0.3$. Use your fitted `volfraction` to predict how strongly
  the low-q intensity is suppressed.
- $Q^* \approx 6/\sigma$ with $\sigma = 2\times$`radius_effective`. Does this
  match the bump position you read off above?

*Physicists:* this is the same $S(Q)$ as for an atomic liquid, with each
colloidal particle playing the role of an atom (slide 24).

In [ ]:
interacting = SANSFitter()
interacting.load_data(str(filename))

# Configure the same sphere form factor, add 'hardsphere', and call
# `get_params()` before configuring radius_effective and volfraction.

# YOUR CODE HERE
raise NotImplementedError('Add, inspect, configure, and fit the structure factor')


In [ ]:
# Report both objective values and the fitted radius,
# effective radius, and volume fraction.

# YOUR CODE HERE
raise NotImplementedError('Compare the two physical explanations')


## Exercise 3 — Interpret and challenge

1. What real-space correlation creates the peak?
2. Why might `radius_effective` differ from the form-factor radius?
3. What assumptions in the hard-sphere model could fail?
4. Try `radius_effective_mode='link_radius'`. Is that constraint supported by
   these data?

Notes for these questions:

- For question 2, think about what surrounds a real colloid: a stabilizer or
  surfactant shell, a hydration layer, or a counter-ion cloud. Charged
  particles repel over longer distances, and slide 24 notes that screened
  charges (Hayter–Penfold/MSA) produce peaks at much lower concentration.
  Is the shell visible to the *form factor* at the given contrast?
- For question 3, the hard-sphere model assumes monodisperse, spherical,
  purely repulsive particles. Attractive interactions *raise* $S(0)$ instead
  (slide 24), and the "decoupling" of $P(Q)$ and $S(Q)$ is an approximation
  for polydisperse or anisotropic particles.
- For question 4, `radius_effective_mode` is an argument of
  `set_structure_factor`, not of `set_param`. Build a new fitter and call
  `set_structure_factor('hardsphere', radius_effective_mode='link_radius')`,
  as on slide 24. Compare the residuals and reduced χ² with your free fit.

In [ ]:
# TODO: explain what changed physically, not only statistically.
interpretation = 'Write your physical interpretation here.'
print(interpretation)


## Exit ticket

Sketch a decision tree for an unfamiliar curve: data checks, simplest form
factor, residual diagnosis, alternative shapes, q-range sensitivity, and only
then interaction models. Name one conclusion that SANS alone cannot establish.

Use the lecture's *Step 0* and *Fitting strategy* (slides 28 and 30) as the
root of your tree: know the composition, concentration, charge and likely
sizes first. Then use the model-free looks (Guinier, power-law slopes,
Kratky, P(r); slides 19, 20 and 25) before model fitting. Bring your tree to
the plenary discussion.

## Further reading

- [SasView `hardsphere` model](https://www.sasview.org/docs/user/models/hardsphere.html): the
  Percus–Yevick structure factor used here.
- [SasView `hayter_msa` model](https://www.sasview.org/docs/user/models/hayter_msa.html): the
  Hayter–Penfold structure factor for charged particles mentioned on slide 24.
- J. S. Pedersen, Adv. Colloid Interface Sci. 70, 171 (1997): the section on
  concentrated systems and the decoupling approximation.